# V2.1-C: Stress confirmatory evaluation (step 7)

This notebook is a text-only reporting aid, like notebooks 11 and 12:
it loads the frozen `v2_stress_protocol.json`, the sealed aggregate
result, and the small published manifest, and prints them together as
one plain-text report. There is no modeling, no plotting, and no
dataframes here -- only reading and printing what the sealed run
published.

Step 7, governed by
`docs/ADR-014-v2-stress-confirmatory-evaluation.md`, is the single
opening of the `stress` partition, 2025-07-01 to 2025-12-31. ADR-014
promotes `stress` from a diagnostic partition to a confirmatory one,
because `test` -- the partition designed to confirm -- was already
consumed by S8 and is forbidden to V2 by ADR-010.

A single pass over the sealed data scores two arms:

- `v2_combined`: the primary arm, stage A overriding the frozen S7
  fallback wherever its margin reaches the calibrated threshold;
- `s7_fallback_alone`: the frozen S7 control, on the exact same rows.

The input files, when present, are:

- `config/v2_stress_protocol.json`: the frozen pre-registration,
  always present, even before the sealed run happens;
- `temp/v2/v2_stress_results.json`: the full aggregate result, written
  only after the one-shot sealed opening;
- `config/v2_stress_results.json`: the small published manifest.

This notebook does not read the unlock token, does not hold it, and
cannot trigger the sealed run by itself. It only renders published
aggregate evidence.

## Why the paired contrast is the primary comparison

`stress` 2025-H2 has a different class mix than the windows V2 was
developed and calibrated on: in novel groups, `money_services` falls
62 percent, `student_loan` falls 44 percent, `credit_reporting` falls
22 percent, and the critical class falls 16 percent relative to
`test`. Because macro-F1 averages nine classes unweighted, an absolute
V2 number on `stress` is not comparable to the S8 number on `test`,
even for the identical model: the same frozen S7, completely
unchanged, already scored 0.339665 critical F1 on `validation` and
0.257843 on `test`, a difference of 0.081822 attributable to the
window alone.

The paired contrast is immune to that drift because both arms are
scored on the identical rows in the same pass. Four gates decide the
outcome, evaluated on the scientific view only: the three absolute
floors carried over unchanged from S8 (macro-F1 at least 0.69,
critical F1 at least 0.2715, critical precision at least 0.20), and
one paired, strict gate -- the V2-combined critical F1 must be
strictly greater than the S7-alone critical F1 on the same rows. With
4 of 4, the status is `CONFIRMED`; otherwise `NOT_CONFIRMED`. The
development paired gain, 0.047234, is a pre-registered expectation,
not a gate.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.v2_import import (
    load_stress_payload,
    render_stress_import_report,
)

In [2]:
STRESS_PROTOCOL_PATH = PROJECT_ROOT / 'config' / 'v2_stress_protocol.json'

protocol_payload = load_stress_payload(STRESS_PROTOCOL_PATH)

In [3]:
STRESS_RESULT_PATH = PROJECT_ROOT / 'temp' / 'v2' / 'v2_stress_results.json'
STRESS_MANIFEST_PATH = PROJECT_ROOT / 'config' / 'v2_stress_results.json'

result_payload = load_stress_payload(STRESS_RESULT_PATH)
manifest_payload = load_stress_payload(STRESS_MANIFEST_PATH)

In [4]:
print(render_stress_import_report(
    result_payload,
    manifest_payload,
    protocol_payload,
))

V2.1-C STRESS CONFIRMATORY EVALUATION

VERDICT
  stage: V2.1-C   adr: ADR-014
  stress_scope: 2025-07-01 to 2025-12-31
  status: NOT_CONFIRMED
  confirmed: False
  deploy: False
  gates_passed: 3 / 4

  *** NOT_CONFIRMED: at least one pre-registered gate failed on the sealed stress partition. ***
  A CONFIRMED verdict never authorizes deployment; deploy is always false.

GATES (SCIENTIFIC VIEW, 4 REQUIRED)
     gate                     observed  limit     strict  verdict
  -  -----------------------  --------  --------  ------  -------
     macro_f1                 0.710748  0.690000  False   PASS   
  !  critical_f1              0.260404  0.271500  False   FAIL   
     critical_precision       0.426070  0.200000  False   PASS   
     paired_critical_f1_gain  0.006455  0.000000  True    PASS   
  ! marks a failed gate.

  *** AT LEAST ONE GATE FAILED. ***

THE PAIRED CONTRAST (PRIMARY, DRIFT-CONTROLLED)
  v2_combined critical_f1:       0.260404
  s7_fallback_alone critical_f1: 0.253949

## Reading the verdict

VERDICT states the status, the confirmed flag, and that `deploy` is
always `false`: a `CONFIRMED` verdict never authorizes deployment,
only that the frozen package passed the four pre-registered gates on
a window it had never seen. THE PAIRED CONTRAST is the
drift-controlled block: the same two arms, the same rows, one opening
of the seal. EXPECTATION CHECK is diagnostic, not a gate, and only
reports whether the observed gain agrees in sign with the
pre-registered development gain.

A `NOT_CONFIRMED` verdict closes the V2 cycle with the published
measure, exactly as S8 closed V1; it does not open a V2.2 on `stress`.
`monitor` 2026 remains sealed and diagnostic either way, and this was
the last independent read available to V2.